In [ ]:
from datasets.packaged_modules.pandas.pandas import Pandas
%load_ext autoreload
%autoreload 2
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
from BaselineModel import BaselineModel
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter


In [ ]:
load_dotenv()
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)
BASE_MAT_DIRECTORY = os.getenv('BASE_MAT_DIRECTORY')


In [ ]:
import shutil
def init_mat_detector_directory(variant_name:str, train_df: pd.DataFrame, test_df: pd.DataFrame):
    variant_directory = os.path.join(BASE_MAT_DIRECTORY, variant_name)
    MAT_NEW_INPUT_DIRECTORY = f'{variant_directory}/trained/input'
    MAT_NEW_OUTPUT_DIRECTORY = f'{variant_directory}/trained/output'
    shutil.copytree('../config/baseline/dic', MAT_NEW_INPUT_DIRECTORY + '/dic', dirs_exist_ok=True)
    os.makedirs(os.path.join(MAT_NEW_INPUT_DIRECTORY, 'origin'), exist_ok=True)
    os.makedirs(MAT_NEW_OUTPUT_DIRECTORY, exist_ok=True)
    for kv in [{'train': train_df}, {'test': test_df}, {'merged': pd.concat([train_df.assign(project='train'), test_df.assign(project='test')])}]:
        for df_name, df in kv.items():
            if df_name == 'merged':
                df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/comments', index=False, header=False)
                df['label'].str.lower().map({'yes': 'SATD', 'no': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/labels', index=False, header=False)
                df['project'].to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/projects', index=False, header=False)
            else:
                df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/data--{df_name}.txt', index=False, header=False)
                df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/label--{df_name}.txt', index=False, header=False)




    MAT_PRETRAINED_INPUT_DIRECTORY = f'{variant_directory}/pretrained/input'
    MAT_PRETRAINED_OUTPUT_DIRECTORY = f'{variant_directory}/pretrained/output'
    os.makedirs(MAT_PRETRAINED_OUTPUT_DIRECTORY, exist_ok=True)
    shutil.copytree('../config/baseline', MAT_PRETRAINED_INPUT_DIRECTORY, dirs_exist_ok=True)

    test_df['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/data--test.txt', index=False, header=False)
    test_df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', index=False, header=False)
    df1 = pd.DataFrame(open('../config/baseline/origin/data--train.txt').read().splitlines(), columns=["text"])
    df2 = test_df[['text']]
    pd.concat([df1, df2])['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/comments', index=False, header=False)

    df1 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--train.txt', header=None, names=['label']).assign(project='train')
    df2 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', header=None, names=['label']).assign(project='test')
    df = pd.concat([df1, df2])
    df['label'].map({'positive': 'SATD', 'negative': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/labels', columns=['label'], index=False, header=False)

    df.to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/projects', columns=['project'], index=False, header=False)
    return variant_directory



In [ ]:
default_detect_directory = init_mat_detector_directory("default", detect_train_df, detect_test_df)

# Potdar Pattern

In [ ]:
pattern_model = BaselineModel('detect', f'pretrained-potdar-Pattern', simple_output_label_converter, "pretrained", "Pattern", default_detect_directory)
pattern_model.fit(detect_train_dataset)
pattern_model.predict(detect_test_dataset, DATASET_NAME)

# Text Mining

In [ ]:
pretrained_tm_model = BaselineModel('detect', f'pretrained-TM', simple_output_label_converter, "pretrained", "TM", default_detect_directory)
pretrained_tm_model.fit(detect_train_dataset)
pretrained_tm_model.predict(detect_test_dataset, DATASET_NAME)

In [ ]:
trained_tm_model = BaselineModel('detect', f'trained-TM', simple_output_label_converter, "trained", "TM", default_detect_directory)
trained_tm_model.fit(detect_train_dataset)
trained_tm_model.predict(detect_test_dataset, DATASET_NAME)

# NLP

In [ ]:
pretrained_nlp_model = BaselineModel('detect', f'pretrained-NLP', simple_output_label_converter,"pretrained", "NLP", default_detect_directory)
pretrained_nlp_model.fit(detect_train_dataset)
pretrained_nlp_model.predict(detect_test_dataset, DATASET_NAME)

In [ ]:
trained_nlp_model = BaselineModel('detect', f'trained-NLP', simple_output_label_converter, "trained", "NLP", default_detect_directory)
trained_nlp_model.fit(detect_train_dataset)
trained_nlp_model.predict(detect_test_dataset, DATASET_NAME)

# MAT

In [ ]:
pretrained_mat_model = BaselineModel('detect', f'pretrained-MAT', simple_output_label_converter, "pretrained", "MAT", default_detect_directory)
pretrained_mat_model.fit(detect_train_dataset)
pretrained_mat_model.predict(detect_test_dataset, DATASET_NAME)

5 Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
df = pd.concat([detect_train_df, detect_test_df])

X = df["text"]
y = df["label"]
groups = df["repository"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    fold_suffix = f"5fcv-{fold+1}"
    print(f'Fold fold_suffix: {len(train_df)} train and {len(test_df)} test samples')

    variant_directory = init_mat_detector_directory(fold_suffix, train_df, test_df)
    trained_nlp_model = BaselineModel('detect', f'trained-NLP-{fold_suffix}', simple_output_label_converter, "trained", "NLP", variant_directory)
    trained_nlp_model.fit(Dataset.from_dict(train_df))
    trained_nlp_model.predict(Dataset.from_pandas(test_df), DATASET_NAME)

    trained_tm_model = BaselineModel('detect', f'trained-TM-{fold_suffix}', simple_output_label_converter, "trained", "TM", default_detect_directory)
    trained_tm_model.fit(Dataset.from_pandas(train_df))
    trained_tm_model.predict(Dataset.from_pandas(test_df), DATASET_NAME)
